# Train the 915M model — one cell

**Before running:** sidebar → **Accelerator: GPU T4 x2**, **Internet: On**.
Then **Add-ons → Secrets** and add `HF_TOKEN` (a *write* token from
huggingface.co → Settings → Access Tokens) — the cell reads it, you never
paste it anywhere.

Run the cell below. It searches the HuggingFace Hub for pretraining text,
downloads a 2 GB budget (a fresh mix each run), builds a tokenizer, measures
real throughput before committing to a step count, trains the 915M `xl`
model with `--momentum 0` (the setting that fits it in 15 GB — plain SGD
momentum would cost an extra 3.7 GB it doesn't have), evaluates it on held-out
text, prints sample generations, and publishes the result to
[huggingface.co/m9cherif3/ai_from_scratch](https://huggingface.co/m9cherif3/ai_from_scratch)
so it survives the session ending.

Checkpoints for this model are 3.5 GB each. The throughput probe skips
validation entirely and its checkpoint directory is deleted before the real
run starts, and the real run keeps only one rotating checkpoint
(`--keep-last-n 1`) — both learned from a run whose probe alone filled
Kaggle's disk through evaluation-triggered saves before the real training run
had written anything of its own. The final step only runs if a checkpoint
actually exists, so a failed run says so plainly instead of claiming success.

Repo: [m9cherif/ai_from_zero](https://github.com/m9cherif/ai_from_zero)
(branch `main-5h8bvh`) — every fix behind this cell (memory, dataset
selection, tokenizer robustness, checkpoint disk safety) lives there, not in
this notebook, so it stays current without re-editing this file.


In [ ]:
# ── Train the 915M model on Kaggle, search-picked data, auto-published ────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")   # Add-ons -> Secrets

%cd /kaggle/working
!rm -rf ai_from_zero
!git clone -q --branch main-5h8bvh --single-branch https://github.com/m9cherif/ai_from_zero.git
%cd /kaggle/working/ai_from_zero
!pip install -q -r requirements.txt huggingface_hub

from myai.core.device import setup, describe
print(describe(setup("auto")))

# 1. Search the Hub for real pretraining text and download to a size budget.
#    No --seed: a different mix of shards each time this runs.
!python scripts/list_datasets.py --limit 60 --scan 1200 --prose-only \
    --out catalogue.json --show 10
!python scripts/fetch_corpus.py --hf-from catalogue.json --skip-gutenberg \
    --budget-mb 2000 --max-per-dataset 4

# 2. Tokenizer.
!python scripts/build_tokenizer.py --type bpe --vocab-size 4096 --data 'data/train/*.txt'

# 3. Measure real throughput before committing to a step count - --steps
#    drives the cosine LR schedule, so guessing it either leaves the rate
#    high at the end or anneals to nothing with hours still on the clock.
#
#    No --val-data here: a checkpoint for this model is 3.5 GB, and this
#    model's default keep_last_n=3 means keep_last_n rotating files plus
#    checkpoint_latest.pt plus checkpoint_best.pt can coexist - up to five
#    files, ~17 GB, from evaluation-triggered "new best" saves alone. One
#    real run's probe validated every 15 steps (eval_every_steps scales with
#    --steps, not with --save-every) and filled Kaggle's disk by itself
#    before the real training run had written a single checkpoint of its
#    own. The probe only needs to measure tokens/s, so it skips validation
#    entirely - --save-every 100000 alone does not prevent this, since
#    validation saves run on their own schedule.
BASE_ARGS = (
    "--preset xl --data data/train --cache-dir output/tokens "
    "--lr 0.05 --dropout 0.0 --optimizer sgd --momentum 0 --grad-checkpoint "
    "--no-save-optimizer --dtype float16 --mixed-precision --flash --device auto"
)
!python scripts/train.py {BASE_ARGS} --batch-size 8 --steps 60 \
    --log-every 10 --save-every 100000 --output /kaggle/working/probe 2>&1 | tee probe.log

import re
_matches = re.findall(r"tokens/s=([\d.]+)", open("probe.log").read())
tokens_per_sec = float(_matches[-1]) if _matches else 800.0
steps = max(500, min(200_000, int(tokens_per_sec * 10.5 * 3600 / (8 * 256))))
print(f"measured {tokens_per_sec:.0f} tokens/s -> training for {steps:,} steps")

# The probe's own checkpoint directory (if it wrote anything before this
# point) is done its job - free the space before the real run needs it.
!rm -rf /kaggle/working/probe

# 4. Train for real. --keep-last-n 1 caps rotation at (1 rotating file +
#    latest + best) x 3.5 GB =~ 10.5 GB, leaving headroom for the corpus and
#    token cache already on disk (checkpoint/manager.py also rotates old
#    files before writing a new one now, not only after, so a save that
#    would have failed on a nearly-full disk gets the space freed first).
!python scripts/train.py {BASE_ARGS} --val-data data/val --batch-size 8 \
    --steps {steps} --keep-last-n 1 \
    --log-every 100 --save-every 1000 --output /kaggle/working/xl

# 5. Score and sample - only if training actually produced a checkpoint.
checkpoint = "/kaggle/working/xl/checkpoint_latest.pt"
if os.path.exists(checkpoint):
    !python scripts/evaluate.py --data data/val --max-batches 200 --device auto \
        --checkpoint {checkpoint}
    !python scripts/chat.py --max-tokens 60 --device auto \
        --checkpoint {checkpoint} \
        --prompt "The old man walked into the" "She said that"

    # 6. Publish. Kaggle deletes /kaggle/working when the session ends, so
    #    this is what makes the run durable - weights, tokenizer, config and
    #    a model card generated from the checkpoint itself land on the Hub.
    !python scripts/push_model.py --checkpoint {checkpoint} --tokenizer output/tokenizer.json
    print("\nDone -> https://huggingface.co/m9cherif3/ai_from_scratch")
else:
    print(f"\nNo checkpoint at {checkpoint} - training did not complete. "
          f"Check the training output above for the actual error before re-running.")
